# SegRNN — Stage 2b: FFT, encoder depth/direction, value embedding, normalization, loss function

Second, independent notebook — not a continuation of `colab_runner.ipynb`.
Kept separate (rather than prepended to the first notebook, which is
already large) so each notebook is a self-contained unit you can run on
its own fresh runtime. Mounts Drive and clones/pulls the repo itself,
same as the first notebook.

Six new, independent Stage 2 strands, each tested only against the same
`RECON_BASELINE` (hardcoded from `results/runs.csv`, defined below) —
same reasoning as every prior strand (see `docs/stage2_new_strands_draft.md`'s
Discussion): stacking an untested idea onto another untested idea would
make it impossible to attribute the result to either one.

- **Part 1** — top-K FFT-magnitude features (`SegRNNFFT`), run **with and
  without** `--power_transform 1` (Yeo-Johnson) — 2 variants x 4 horizons.
- **Part 2** — encoder depth: stacked GRU, `--encoder_layers 2` (`SegRNNDeepEncoder`).
- **Part 3** — encoder direction: bidirectional encoder + separate decode
  cell (`SegRNNBidir`) — the heaviest strand here, see its own docstring.
- **Part 4** — value embedding: within-segment `Conv1d` instead of one
  `Linear` over the whole segment (`SegRNNConvEmbed`).
- **Part 5** — normalization: RevIN with `affine=True` (`SegRNNRevINAffine`),
  vs. both the reconstruction and the already-tested `affine=False` RevIN.
- **Part 6** — loss function: Huber and blended MSE+MAE, each **with and
  without** Yeo-Johnson — 4 combinations, scoped to 2 horizons (336, 720)
  to keep the run count sane (same compute-budget reasoning as Part 6 of
  the first notebook).

Total: ~32 new training runs across the six parts — run Parts
independently across sessions if needed; none of them depend on each other.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

Same pattern as the first notebook's setup cell: `run_horizon` launches
`run_longExp.py` as a subprocess, streams output live, parses the final
`mse:.., mae:..` line. Generalized here with `**extra_flags` so new CLI
flags (`--encoder_layers`, `--conv_kernel_size`, `--loss`, ...) don't
need their own named parameter each.

In [ ]:
import os, sys, re, csv, subprocess, datetime, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]

# Reconstruction baseline (SegRNN, d_model=512, seed=2024), from results/runs.csv --
# the same comparison point every Stage 2 strand in this project uses. Hardcoded so
# this notebook doesn't depend on colab_runner.ipynb having been run first.
RECON_BASELINE = {
    'mse': {96: 0.3510, 192: 0.3925, 336: 0.4233, 720: 0.4657},
    'mae': {96: 0.3925, 192: 0.4142, 336: 0.4327, 720: 0.4719},
}

# Already-measured results from colab_runner.ipynb's Parts 0a/8, for context in
# this notebook's own comparisons (not re-derived here, just referenced).
YEOJOHNSON_ALONE = {
    'mse': {96: 0.3250, 192: 0.3640, 336: 0.4038, 720: 0.4489},
    'mae': {96: 0.4085, 192: 0.4368, 336: 0.4616, 720: 0.4926},
}
REVIN_NOAFFINE = {
    'mse': {96: 0.3673, 192: 0.4068, 336: 0.4376, 720: 0.4801},
    'mae': {96: 0.4003, 192: 0.4224, 336: 0.4397, 720: 0.4776},
}

os.makedirs('results/figures', exist_ok=True)


def run_horizon(model, pred_len, seed=None, d_model=512, revin=None, power_transform=None,
                 loss=None, huber_delta=None, blend_alpha=None, **extra_flags):
    """Launch run_longExp.py for one (model, horizon), stream its output
    live, and parse the final 'mse:X, mae:Y, ms/sample:Z' line it prints.
    Returns (mse, mae, ms_per_sample). extra_flags passes through arbitrary
    --flag value pairs (e.g. encoder_layers=2, conv_kernel_size=5, fft_k=32)."""
    model_id = (f'ETTh1_720_{pred_len}'
                + (f'_seed{seed}' if seed is not None else '')
                + (f'_dm{d_model}' if d_model != 512 else '')
                + (f'_revin{revin}' if revin is not None else '')
                + (f'_pt{power_transform}' if power_transform is not None else '')
                + (f'_{loss}' if loss is not None else ''))
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', 'ETTh1',
        '--root_path', './dataset/', '--data_path', 'ETTh1.csv',
        '--features', 'M', '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', '24', '--enc_in', '7', '--d_model', str(d_model),
        '--dropout', '0.1', '--rnn_type', 'gru', '--dec_way', 'pmf', '--channel_id', '1',
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', '64', '--learning_rate', '0.0003',
    ]
    if seed is not None:
        cmd += ['--random_seed', str(seed)]
    if revin is not None:
        cmd += ['--revin', str(revin)]
    if power_transform is not None:
        cmd += ['--power_transform', str(power_transform)]
    if loss is not None:
        cmd += ['--loss', loss]
    if huber_delta is not None:
        cmd += ['--huber_delta', str(huber_delta)]
    if blend_alpha is not None:
        cmd += ['--blend_alpha', str(blend_alpha)]
    for flag, val in extra_flags.items():
        cmd += [f'--{flag}', str(val)]

    print(f'\n{"="*70}\n{model}  H={pred_len}'
          + (f'  seed={seed}' if seed is not None else '')
          + (f'  revin={revin}' if revin is not None else '')
          + (f'  power_transform={power_transform}' if power_transform is not None else '')
          + (f'  loss={loss}' if loss is not None else '')
          + (f'  {extra_flags}' if extra_flags else '') + f'\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{model} H={pred_len} failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+), ms/sample:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae/ms-per-sample in output for {model} H={pred_len}')
    return float(m.group(1)), float(m.group(2)), float(m.group(3))


# dataviz-validated categorical palette, fixed order, this notebook's own labels
COLORS = {
    'Reconstruction': '#008300',
    'FFT (raw)': '#2a78d6', 'FFT + Yeo-Johnson': '#a35a00',
    'Deep Encoder': '#9c2c8f', 'Bidirectional': '#e34948',
    'Conv Embed': '#0a8fa3',
    'RevIN (no affine)': '#4a3aa7', 'RevIN (affine)': '#c9a227',
}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, save_path=None):
    """series: list of (label, {horizon: value}), in display order.
    Always creates a brand-new figure."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(f'SegRNN on ETTh1 — {metric_name.upper()}', color=INK_PRIMARY, fontsize=13, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.14),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')

## Part 1 — top-K FFT-magnitude features (`SegRNNFFT`), with and without Yeo-Johnson

Injects the K largest-magnitude FFT bins of the raw look-back window
(per channel) into `h_n` before decoding — same injection point
`SegRNNRocket` used, a different (and more classically time-series-
native) feature source. Run both with `--power_transform 0` and `1`:
Yeo-Johnson is a nonlinear pointwise transform, so it does not commute
with the Fourier transform — FFT features computed on Yeo-Johnson-
transformed values are genuinely different from FFT features on raw
values, not a redundant test. See `models/SegRNNFFT.py`'s docstring.

In [ ]:
fft_results = {}
for pt in [0, 1]:
    for h in HORIZONS:
        fft_results[(pt, h)] = run_horizon('SegRNNFFT', h, power_transform=pt)

fft_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('FFT (raw)', {h: fft_results[(0, h)][0] for h in HORIZONS}),
    ('FFT + Yeo-Johnson', {h: fft_results[(1, h)][0] for h in HORIZONS}),
]
fft_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('FFT (raw)', {h: fft_results[(0, h)][1] for h in HORIZONS}),
    ('FFT + Yeo-Johnson', {h: fft_results[(1, h)][1] for h in HORIZONS}),
]
display(make_table('mse', fft_series_mse))
display(make_table('mae', fft_series_mae))
plot_metric('mse', fft_series_mse, save_path='results/figures/fft_comparison_mse.png')
plot_metric('mae', fft_series_mae, save_path='results/figures/fft_comparison_mae.png')

print('For context -- Yeo-Johnson alone (colab_runner.ipynb Part 0a), no FFT:')
for h in HORIZONS:
    print(f"  H={h}: MSE={YEOJOHNSON_ALONE['mse'][h]:.4f}, MAE={YEOJOHNSON_ALONE['mae'][h]:.4f}")

## Part 2 — encoder depth: stacked GRU (`SegRNNDeepEncoder`)

`--encoder_layers 2` instead of the original single-layer GRU encoder —
a different axis from the paper's own segment-length ablation (which
varies recurrent *steps*, not stacked *layers*; see the model's own
docstring). Given the pattern across every Stage 2 strand tested so far
(added capacity has consistently regressed, AIC/BIC formally rejects
even `d_model=512`), go in expecting skepticism, not a free win.

In [ ]:
depth_results = {}
for h in HORIZONS:
    depth_results[h] = run_horizon('SegRNNDeepEncoder', h)

depth_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Deep Encoder', {h: depth_results[h][0] for h in HORIZONS}),
]
depth_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Deep Encoder', {h: depth_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', depth_series_mse))
display(make_table('mae', depth_series_mae))
plot_metric('mse', depth_series_mse, save_path='results/figures/deepencoder_comparison_mse.png')
plot_metric('mae', depth_series_mae, save_path='results/figures/deepencoder_comparison_mae.png')

## Part 3 — encoder direction: bidirectional (`SegRNNBidir`)

`CLAUDE.md`'s own unimplemented improvement B. Whole look-back window is
past data, so bidirectional encoding isn't leakage. Structurally the
heaviest strand tested in this project — a bidirectional encoder can't
reuse SegRNN's shared-cell trick for decoding, so this model adds a full
second (decode-only) GRU on top of the bidirectional encoder. See the
model's own docstring for the full accounting of added capacity.

In [ ]:
bidir_results = {}
for h in HORIZONS:
    bidir_results[h] = run_horizon('SegRNNBidir', h)

bidir_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Bidirectional', {h: bidir_results[h][0] for h in HORIZONS}),
]
bidir_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Bidirectional', {h: bidir_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', bidir_series_mse))
display(make_table('mae', bidir_series_mae))
plot_metric('mse', bidir_series_mse, save_path='results/figures/bidir_comparison_mse.png')
plot_metric('mae', bidir_series_mae, save_path='results/figures/bidir_comparison_mae.png')

## Part 4 — value embedding: within-segment `Conv1d` (`SegRNNConvEmbed`)

Replaces the single `Linear(seg_len -> d_model)` with a `Conv1d` whose
kernel (`--conv_kernel_size`, default 5) is narrower than the segment,
so it slides within each segment instead of consuming it in one shot —
a genuinely different inductive bias, not a relabeling of the same
computation (see the model's docstring for why kernel=seg_len would be).
`CLAUDE.md` already flags this as the higher-risk of its two listed
architecture candidates, citing ISMRNN's reported conv regressions.

In [ ]:
conv_results = {}
for h in HORIZONS:
    conv_results[h] = run_horizon('SegRNNConvEmbed', h)

conv_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Conv Embed', {h: conv_results[h][0] for h in HORIZONS}),
]
conv_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Conv Embed', {h: conv_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', conv_series_mse))
display(make_table('mae', conv_series_mae))
plot_metric('mse', conv_series_mse, save_path='results/figures/convembed_comparison_mse.png')
plot_metric('mae', conv_series_mae, save_path='results/figures/convembed_comparison_mae.png')

## Part 5 — normalization: RevIN with `affine=True` (`SegRNNRevINAffine`)

RevIN with `affine=False` already regressed at every horizon
(`colab_runner.ipynb` Part 8, `docs/stage2_revin_attn_draft.md` strand
3). `affine=True` adds back RevIN's learnable per-channel post-
normalization scale/shift that configuration disabled — explicitly
flagged as the untested variant in that write-up's limitations section.
Compared against both the reconstruction and the already-measured
`affine=False` numbers.

In [ ]:
revin_affine_results = {}
for h in HORIZONS:
    revin_affine_results[h] = run_horizon('SegRNNRevINAffine', h, revin=1)

revin_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('RevIN (no affine)', REVIN_NOAFFINE['mse']),
    ('RevIN (affine)', {h: revin_affine_results[h][0] for h in HORIZONS}),
]
revin_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('RevIN (no affine)', REVIN_NOAFFINE['mae']),
    ('RevIN (affine)', {h: revin_affine_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', revin_series_mse))
display(make_table('mae', revin_series_mae))
plot_metric('mse', revin_series_mse, save_path='results/figures/revinaffine_comparison_mse.png')
plot_metric('mae', revin_series_mae, save_path='results/figures/revinaffine_comparison_mae.png')

## Part 6 — loss function: Huber and blended MSE+MAE, with and without Yeo-Johnson

Targets the same MSE-vs-MAE trade-off Yeo-Johnson exposed
(`docs/stage2_new_strands_draft.md` strand 5: Yeo-Johnson improved MSE
3.6-7.4% while worsening MAE 4.1-6.7% at every horizon), but at the loss
function instead of the preprocessing step. `--loss huber` (delta=1.0)
is less sensitive to outliers than MSE but still smoother than MAE near
zero; `--loss blend` is `alpha*MSE + (1-alpha)*MAE` (alpha=0.5). Both
tried with `--power_transform 0` and `1` — 4 combinations. Model is
plain `SegRNN`, no new architecture file needed. Scoped to 2 horizons
(336, 720) to keep the run count reasonable (same reasoning as the first
notebook's Part 6 seed-variance check and Part 0c ensembling).

In [ ]:
LOSS_HORIZONS = [336, 720]
loss_results = {}
for loss_name in ['huber', 'blend']:
    for pt in [0, 1]:
        for h in LOSS_HORIZONS:
            loss_results[(loss_name, pt, h)] = run_horizon('SegRNN', h, loss=loss_name, power_transform=pt)

rows = []
for h in LOSS_HORIZONS:
    row = {'Horizon': h, 'Reconstruction MSE': RECON_BASELINE['mse'][h], 'Reconstruction MAE': RECON_BASELINE['mae'][h]}
    for loss_name in ['huber', 'blend']:
        for pt in [0, 1]:
            mse, mae, _ = loss_results[(loss_name, pt, h)]
            tag = f'{loss_name}{"+YJ" if pt else ""}'
            row[f'{tag} MSE'] = round(mse, 4)
            row[f'{tag} MAE'] = round(mae, 4)
    rows.append(row)
loss_df = pd.DataFrame(rows).set_index('Horizon')
display(loss_df)

print('Delta vs. Reconstruction, MSE:')
for h in LOSS_HORIZONS:
    for loss_name in ['huber', 'blend']:
        for pt in [0, 1]:
            mse, mae, _ = loss_results[(loss_name, pt, h)]
            tag = f'{loss_name}{"+YJ" if pt else ""}'
            d_mse = (mse / RECON_BASELINE['mse'][h] - 1) * 100
            d_mae = (mae / RECON_BASELINE['mae'][h] - 1) * 100
            print(f'  H={h} {tag:12s}: MSE {d_mse:+.1f}%, MAE {d_mae:+.1f}%')

## Optional — save results back into the repo

Appends this notebook's rows to `results/runs.csv` (reads the existing
file first, doesn't overwrite `colab_runner.ipynb`'s rows). Commit/push
left commented out on purpose — review `git status`/`git diff` first.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []

for pt in [0, 1]:
    for h in HORIZONS:
        mse, mae, ms = fft_results[(pt, h)]
        rows.append([f'SegRNNFFT_ETTh1_{h}_pt{pt}_{ts}', ts, 'SegRNNFFT', 'ETTh1', h, 720, 24, 512, 2024,
                     f'seg_len=24;d_model=512;fft_k=32;power_transform={pt}', mse, mae, '', '', '', '',
                     'top-K FFT magnitude features' + (' + Yeo-Johnson' if pt else '')])

if 'depth_results' in dir():
    for h, (mse, mae, ms) in depth_results.items():
        rows.append([f'SegRNNDeepEncoder_ETTh1_{h}_{ts}', ts, 'SegRNNDeepEncoder', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512;encoder_layers=2', mse, mae, '', '', '', '', 'stacked GRU encoder'])

if 'bidir_results' in dir():
    for h, (mse, mae, ms) in bidir_results.items():
        rows.append([f'SegRNNBidir_ETTh1_{h}_{ts}', ts, 'SegRNNBidir', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512', mse, mae, '', '', '', '', 'bidirectional encoder + separate decode cell'])

if 'conv_results' in dir():
    for h, (mse, mae, ms) in conv_results.items():
        rows.append([f'SegRNNConvEmbed_ETTh1_{h}_{ts}', ts, 'SegRNNConvEmbed', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512;conv_kernel_size=5', mse, mae, '', '', '', '', 'conv value embedding'])

if 'revin_affine_results' in dir():
    for h, (mse, mae, ms) in revin_affine_results.items():
        rows.append([f'SegRNNRevINAffine_ETTh1_{h}_{ts}', ts, 'SegRNNRevINAffine', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512;revin=1;affine=1', mse, mae, '', '', '', '', 'RevIN affine=True'])

if 'loss_results' in dir():
    for (loss_name, pt, h), (mse, mae, ms) in loss_results.items():
        rows.append([f'SegRNN_ETTh1_{h}_{loss_name}_pt{pt}_{ts}', ts, 'SegRNN', 'ETTh1', h, 720, 24, 512, 2024,
                     f'seg_len=24;d_model=512;loss={loss_name};power_transform={pt}', mse, mae, '', '', '', '',
                     f'{loss_name} loss' + (' + Yeo-Johnson' if pt else '')])

existing = pd.read_csv('results/runs.csv') if os.path.exists('results/runs.csv') else pd.DataFrame(columns=RUNS_CSV_HEADER)
new_df = pd.DataFrame(rows, columns=RUNS_CSV_HEADER)
combined = pd.concat([existing, new_df], ignore_index=True)
combined.to_csv('results/runs.csv', index=False)
print(f'appended {len(rows)} rows to results/runs.csv (total {len(combined)})')

!git add results/runs.csv results/figures/
!git status
# review the diff above, then when ready:
# !git commit -m "Update results: FFT, encoder depth/direction, conv embed, RevIN affine, loss function"
# !git push origin main